# Setup

In [1]:
import pandas as pd
import requests

In [2]:
from io import StringIO

url = 'https://raw.githubusercontent.com/pythonhk/20260530-data-workshop/refs/heads/main/data/train_features.csv'
response = requests.get(url)
df = pd.read_csv(StringIO(response.text))

In [3]:
df

,record_id,route_id,origin_station,destination_station,district,encoded_transport,fare_hkd,distance_km,scheduled_duration_min,hour_of_day,...,session_id,ticket_reference,device_id_hash,distance_m,fare_hkd_rounded,route_group,is_peak_hour,service_note,ops_comment_code,route_label_variant
0,R00001,RT075,CentralStation,Admiralty,Central and Western,tram__local-standard__CTB;backup=CTB,NaN,NaN,NaN,18,...,S2031111535,TKT-331000,dev-c58114a0,6760,10,group-0,1,weather watch,OPS-D,Route-075
1,R00002,RT063,Kennedy Town,Admiralty,Central and Western,bus_local_standard_KMB,missing,2.01,14.0,7,...,S9969218457,TKT-890486,dev-cd6a3799,2010,20,group-0,1,maintenance escort,OPS-SYS,ROUTE 063
2,R00003,RT005,Central Station,Sha Tin,Sha Tin,Bus_LOCAL_Standard_K.M.B.,NaN,7.91,NaN,18,...,S1632419629,TKT-484610,dev-00faef30,7910,13,group-0,1,platform crowding,OPS-B,Route-005
3,R00004,RT030,Tsim Sha Tsui East,Sha Tin,Sha Tin,bus_EXPRESS_Standard_Kowloon Motor Bus,NaN,NaN,NaN,20,...,S5929138623,TKT-175440,dev-40babff7,7560,17,group-0,0,priority boarding,OPS-C,ROUTE 030
4,R00005,RT060,Tsim Sha Tsui,Causeway Bay,Yau Tsim Mong,tramexppremCTB,21.6,6.38,0.7 hrs,6,...,S4658130369,TKT-808744,dev-ba75cac0,6380,22,group-0,0,platform crowding,OPS-C,ROUTE 060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8402,R05292,RT039,Central Station,Sha Tin,Wan Chai,bus_local_standard_K.M.B.,2500 HKD,NaN,NaN,19,...,S2057437999,TKT-675218,dev-39237a06,2410,7,group-0,1,weather watch,OPS-D,Route-039
8403,R07727,RT026,Central.Station,tst_east,Central and Western,bus_express_premium_K.M.B._v7,2500 HKD,unknown,T29,23,...,S5504502228,TKT-425070,dev-97c6b2c3,5190,23,group-0,0,platform crowding,OPS-D,Route-026
8404,R00281,RT032,Sha Tin,Kennedy Town,Tsuen Wan,bus-XHBR_local_standard_K.M.B.,missing,NaN,NaN,10,...,S3188472060,TKT-392404,dev-226ebc8f,1470,7,group-0,0,platform crowding,OPS-D,ROUTE 032
8405,R04687,RT047,Audit Hub,Audit Hub,Tsuen Wan,maintenance_shift,HK$0.5,0.5,2.0,6,...,S5996608331,TKT-373775,dev-5c83194b,900000,9000,group-0,0,test record,OPS-SYS,Route-047


# Numeric Warm-up

## fare_hkd

In [4]:
# use unique to observe the pattern
df['fare_hkd'].unique()

<StringArray>
[        nan,   'missing',      '21.6',       '0,0',    'HK$0.5', '400.0 HKD',
       '1.0',  '8.5-13.5',    '2500.0',   '1.0 HKD',
 ...
       '9,1',   'HK$16.3', '12.9-17.9',   'HK$20.3',  '9.3 fare', 'fare=10.5',
      '11,2',   '9.5 HKD',      '3.95',  '2500 HKD']
Length: 1003, dtype: str

In [5]:
# always make a copy on the original column first to prevent re-loading
# you can use the original one for cross-check & prevent re-load of data
df['fare_hkd_cleaning'] = df['fare_hkd']

In [6]:
# say found the pattern with string - suppose it should be a numeric column
# then we need to check if there's any pattern we can find
df[df['fare_hkd_cleaning'].str.contains('[a-zA-Z]', na=False)]['fare_hkd_cleaning'].unique()

<StringArray>
[  'missing',    'HK$0.5', '400.0 HKD',   '1.0 HKD',  'fare=2.0',    'HK$6.8',
   '2.0 HKD',    'HK$6.3',    'HK$9.3', '500.0 HKD',
 ...
  '18.6 HKD',  '11.0 HKD',   'HK$10.4', 'fare=15.2',   'HK$16.3',   'HK$20.3',
  '9.3 fare', 'fare=10.5',   '9.5 HKD',  '2500 HKD']
Length: 401, dtype: str

In [7]:
# first approach - do it one by one which is much safer - won't impact other data accidentally
# say I saw ' HKD' & 'fare=' & ' fare' first
pattern_needed_to_clean = [' HKD', 'fare=', ' fare'] # put the pattern you found in the above cell

for pattern in pattern_needed_to_clean:
    # set up the condition for filtering & assigning
    condition = df['fare_hkd_cleaning'].str.contains(pattern)
    
    # .loc can also be for assigning the new value to those row filtered
    # in this case, the pattern will be replace by '' to remove it
    df.loc[condition, 'fare_hkd_cleaning'] = df[condition]['fare_hkd_cleaning'].str.replace(pattern, '')
    
# after this cell finished, you can re-run the above cell
# see if there's other pattern you would like to add to the pattern_needed_to_clean until it's all done.
    
# down side: you may need to add the pattern again in future if there's some new corrupted data

In [8]:
# second approach - using regex to work on the data in batch
# directly use the regex to extra the first decimal data & replace it
# say all string related row
condition = df['fare_hkd_cleaning'].str.contains('[a-zA-Z]', na=False)

# didn't do the assign here but you can observe the cell output
df[condition]['fare_hkd_cleaning'].str.extract(r'([\d.]+)', expand=False)

# down side: you better polish the conditional filter as there's some chances to corrupt other rows 

1       NaN
13      0.5
39      NaN
56      6.8
74      6.3
       ... 
8400    0.5
8401    0.5
8404    NaN
8405    0.5
8406    0.5
Name: fare_hkd_cleaning, Length: 898, dtype: str

# Column Drift

In [9]:
df.columns

Index(['record_id', 'route_id', 'origin_station', 'destination_station',
       'district', 'encoded_transport', 'fare_hkd', 'distance_km',
       'scheduled_duration_min', 'hour_of_day', 'day_of_week', 'is_holiday',
       'weather_condition', 'country_code', 'internal_batch_id', 'survey_code',
       'campaign_tag', 'device_theme', 'promo_bucket', 'random_zone_code',
       'passenger_token', 'session_id', 'ticket_reference', 'device_id_hash',
       'distance_m', 'fare_hkd_rounded', 'route_group', 'is_peak_hour',
       'service_note', 'ops_comment_code', 'route_label_variant',
       'fare_hkd_cleaning'],
      dtype='str')

In [10]:
# Upon the hints from your bad colleagues, R03501 - R04500 weather & country was swapped
# Usually, if something happened on the setting from other side, it should be appeared in batch
# Date time column should be one to help

# This time it's record_id & we can try to convert it into number to help filtering
# Same logic can be applied to datetime column

# better to remove the duplicates first
df.drop_duplicates('record_id').reset_index(drop = True) # Own practice to always clean up the index

# Turn the record_id into numeric for your filtering
df['id'] = df['record_id'].str.replace('R','').astype(int)

In [11]:
# set-up the condition to screen it.
# it can be apply to datetime column
# Say you want to check the data for Jan 2026 only
# (df['date'] >= '2026-01-01') & (df['date'] < '2026-02-01') -> '>=' + '<' is recommended if you forgot the end date for the month
condition = (df['id'] >= 3501) & (df['id'] <= 4500)

In [12]:
# Checking the unique value on weather condition and you can see most of it are country related data
df[condition]['weather_condition'].unique()

<StringArray>
[    'geo::hkg',          '852',           'HK',       'legacy',
          'hkg', 'territory-hk',            nan,   'geo-hkg-v2',
           'CN',      'hk-zone',           'MO',     'geo::852',
     'heavy-rn',      'Unknown',        'SUNNY']
Length: 15, dtype: str

In [13]:
# Checking the unique value on country and you can see most of it are weather related data
df[condition]['country_code'].unique()

<StringArray>
[         'sunny',         'cloudy',            'clr',              nan,
        'Unknown',     'Heavy Rain',          'Sunny',     'wx.cld.ops',
           'wx_s',         'rain??',         'Cloudy',      'HeavyRain',
            'TBD',         'CLOUDY', 'wx.hr.critical',       'heavy-rn',
          'SUNNY',           'Rain',    'wx.rn.alert',           'rain',
     'HEAVY_RAIN']
Length: 21, dtype: str

In [14]:
# Then you can switch it & it will be easier to do the cleaning for the whole column now
df.loc[condition,'weather_condition'] , df.loc[condition,'country_code'] = df[condition]['country_code'],df[condition]['weather_condition']

In [15]:
# Checking the unique value after switching & it looks better now although you sill need to clean it.
df[condition]['weather_condition'].unique()

<StringArray>
[         'sunny',         'cloudy',            'clr',              nan,
        'Unknown',     'Heavy Rain',          'Sunny',     'wx.cld.ops',
           'wx_s',         'rain??',         'Cloudy',      'HeavyRain',
            'TBD',         'CLOUDY', 'wx.hr.critical',       'heavy-rn',
          'SUNNY',           'Rain',    'wx.rn.alert',           'rain',
     'HEAVY_RAIN']
Length: 21, dtype: str

In [16]:
# Checking the unique value after switching & it looks better now although you sill need to clean it.
df[condition]['country_code'].unique()

<StringArray>
[    'geo::hkg',          '852',           'HK',       'legacy',
          'hkg', 'territory-hk',            nan,   'geo-hkg-v2',
           'CN',      'hk-zone',           'MO',     'geo::852',
     'heavy-rn',      'Unknown',        'SUNNY']
Length: 15, dtype: str

In [17]:
# Why these have to be set in your pipeline?
# Coz sometime people will deliver the data with a specific date range & you should always backfill for certain period
# of data. In this case, you won't be do the clean by once, say if the data included past 30 days & you should keep your
# code in 30 days for handling.

In [18]:
# You can apply the same logic for the second swap on R5501 to R6500
# or the encoded_swap when you are cleaning it

# Set up the reference for value check

In [19]:
# Always good to set up some reference for value check especially you know how the value should looks like
# Take 'day_of_week' as an example - from the PPT you know that only value should be `Mon`, `Tue`, `Wed`, `Thu`, `Fri`, `Sat`, `Sun`
# Let set up a list for checking - You can build an enum class as well.
DAY_OF_WEEK_VALID_LIST = {'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'}

In [20]:
# you can use it to check if there's any pattern not fullfill the valid set
# then, you can check the non-valid value & how you gonna fix it.
df[~df['day_of_week'].isin(DAY_OF_WEEK_VALID_LIST)]['day_of_week'].unique()

<StringArray>
['tue', 'mon', 'fri', 'thu', 'sat', 'sun', 'wed']
Length: 7, dtype: str

## Url Encoding Reference

Reference for Issues across all the fields - Fix the big issues first.

Ref: https://www.w3schools.com/tags/ref_urlencode.ASP

In [21]:
# Sometime, some issues could impact the whole dataset
# For example, like the url encoding issue. In this case, it's something you better fix it first
# Once these big issues is fixed, it will narrow down other issue as well
# Using origin_station as an example
df['origin_station'].value_counts().head(15)
# You will see some %20 in the data & it's blocking you to view the data in a normal way.

origin_station
Kennedy Town         683
Central Station      672
Sha Tin              654
Admiralty            391
Tsuen Wan            310
Tsim Sha Tsui        296
North Point          283
Wan Chai             247
Causeway Bay         224
Mong Kok             221
Workshop             149
Audit Hub            143
Depot                134
Tsim%20Sha%20Tsui    103
Wan%20Chai            92
Name: count, dtype: int64

In [22]:
# Then, you can build a dict for the fixing.
# Of coz, there's some libs can help & you can do some research on it
ENCODED_FIX = {
    '%20' : ' '
}

for encoded, decoded in ENCODED_FIX.items():
    df['origin_station'] = df['origin_station'].str.replace(encoded, decoded)

In [23]:
# Now, you can see the value is much clear and easier for you to clean it!
df['origin_station'].value_counts().head(15)

origin_station
Kennedy Town       767
Central Station    747
Sha Tin            735
Tsim Sha Tsui      399
Tsuen Wan          394
Admiralty          391
North Point        358
Wan Chai           339
Causeway Bay       304
Mong Kok           298
Workshop           149
Audit Hub          143
Depot              134
North Pt            86
Central Stn         76
Name: count, dtype: int64

In [24]:
# Always think of the big problem first before going into the small/ detail problem
# Then, the data cleaning will be easier!!!

# Naming Convention Application

In [25]:
# from the naming convention definition
# we can see there's a rules with the delimiter
# <transport_type[-detail]>_<mode>_<service_level>_<operator>

In [26]:
# In this case, you can see that it should contains 3 '_' which is the main delimiter
# then, you can filter out those at least with 3 '_' row first
condition = df['encoded_transport'].str.count('_') == 3

# You already can filter out part of the row that can be handled easily
df[condition]['encoded_transport']

1                         bus_local_standard_KMB
2                      Bus_LOCAL_Standard_K.M.B.
3         bus_EXPRESS_Standard_Kowloon Motor Bus
5             TRAM_LOCAL_Standard_ctb;backup=CTB
6                         KMB_bus_standard_local
                          ...                   
8400                     tram_standard_local_CTB
8401                  ferry_express_premium_HKKF
8402                   bus_local_standard_K.M.B.
8404              bus-XHBR_local_standard_K.M.B.
8406    FERRY_Local_Standard_HK Ferry;backup=KMB
Name: encoded_transport, Length: 5967, dtype: str

In [27]:
# then, you can use spliting method to check the value for each field
# say you can work on the first column to omit the issue one by one
df[condition]['encoded_transport'].str.split('_', expand = True)

,0,1,2,3
1,bus,local,standard,KMB
2,Bus,LOCAL,Standard,K.M.B.
3,bus,EXPRESS,Standard,Kowloon Motor Bus
5,TRAM,LOCAL,Standard,ctb;backup=CTB
6,KMB,bus,standard,local
...,...,...,...,...
8400,tram,standard,local,CTB
8401,ferry,express,premium,HKKF
8402,bus,local,standard,K.M.B.
8404,bus-XHBR,local,standard,K.M.B.


In [28]:
# it's good practice to use another temp dataframe to get the value
# & get back to the main code later
temp = df[condition]['encoded_transport'].str.split('_', expand = True)

# then you can reuse the technique mentioned above to work on it 
# to narrow down the data with issue
VALID_TRANSPORT = {'bus', 'tram', 'ferry'}
temp = temp[~temp[0].isin(VALID_TRANSPORT)]
temp[0].unique()

# trust me: there's should not be that much corrupted have to be handled in real world
# it's corrupted like due to workshop purpose only
# effciency will be higher for this step if you fix some of the delimiter issues!

<StringArray>
[                           'Bus',                           'TRAM',
                            'KMB',                       'standard',
                    'bus-airport',                           'Tram',
             'Ferry-CrossHarbour',                      'bus-NIGHT',
                     'meta46:bus',               'Bus-crossharbour',
 ...
               'meta52:tram-XHBR',              'meta254:bus-night',
        'meta[route=58;value=bus',       'meta[route=732;value=bus',
 'meta[route=901;value=bus-night',                    'meta356:bus',
                    'meta800:bus',       'meta[route=879;value=bus',
  'meta[route=659;value=bus-XHBR',                    'meta592:bus']
Length: 230, dtype: str

# Maping value with majority

In [29]:
# This is more about a mapping field, say you already know the value
# of this field is based on another field, say you know district is based on origin_station

# In the usual case, the cleaned data should have the highest count
# if that's not the case, it should be a big issue but it can be discovered easily
df['origin_station'].value_counts()

# take Kennedy Town as an example

origin_station
Kennedy Town       767
Central Station    747
Sha Tin            735
Tsim Sha Tsui      399
Tsuen Wan          394
                  ... 
TSIM.SHA.TSUI        1
 shatin              1
 MONG KOK            1
CEN Station          1
KENNEDY TOWN         1
Name: count, Length: 335, dtype: int64

In [30]:
# you can see that the majority district is 'Central and Western'
df[df['origin_station'] == 'Kennedy Town']['district'].value_counts()

district
Central and Western    434
Sha Tin                222
Wan Chai                36
Eastern                 28
Yau Tsim Mong           27
Tsuen Wan               20
Name: count, dtype: int64

In [31]:
# then, you can use the idxmax method to obtain the value with most count
df[df['origin_station'] == 'Kennedy Town']['district'].value_counts().idxmax()

'Central and Western'

In [32]:
# Then, you can try to use such techniques to help with the mapping

# STATION_VALID_DICT = {'Kennedy Town', 'Sha Tin', ...}
# for station in STATION_VALID_DICT:
#     condition = df['origin_station'] == station
#     major_value = df[condition]['district'].value_counts().idxmax()
#     condition = (df['origin_station'] == station) & (df['district'] != major_value)
#     df.loc[condition, 'district'] = major_value

# If you clean the origin_station column properly
# You can even make it more dynamic instead of fixing with known value
# but you have to keep monitoring it just in case there's an error value became the majority

# .. to be continued